In [ ]:
import os, sys, json, shutil, re, textwrap, glob
from pathlib import Path

# ── Kaggle paths ──────────────────────────────────────────────────────────────
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORK  = Path('/kaggle/working')
EXPORT_DIR   = KAGGLE_WORK / 'meme_model_export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset paths ─────────────────────────────────────────────────────────────
MEMOTION_DIR = KAGGLE_INPUT / 'memotion-dataset-7k'
REDDIT_DIR   = KAGGLE_INPUT / 'reddit-memes-dataset'
TEMPLATE_DIR = KAGGLE_INPUT / 'meme-templates'

print('📂 Datasets found in /kaggle/input:')
for p in sorted(KAGGLE_INPUT.iterdir()):
    status = '✅' if p.is_dir() else '📄'
    print(f'   {status} {p.name}')

print(f'\n📁 Export dir: {EXPORT_DIR}')
print(f'📌 Memotion available: {MEMOTION_DIR.exists()}')
print(f'📌 Reddit memes available: {REDDIT_DIR.exists()}')
print(f'📌 Templates available:  {TEMPLATE_DIR.exists()}')

In [ ]:
# Kaggle already ships torch+cuda — only install extras
!pip install -q transformers==4.40.0 accelerate bitsandbytes sentencepiece
!pip install -q scikit-learn gradio pillow requests

import torch
print(f'✅ PyTorch  {torch.__version__}')
print(f'✅ CUDA     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU:   {torch.cuda.get_device_name(0)}')
    print(f'   VRAM:  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
import pandas as pd
import numpy as np
import requests
from PIL import Image
from io import BytesIO
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MEME_TEMPLATES = []   # {'name': str, 'path': str|None, 'url': str|None, 'source': str}

# ── Source 1: meme-templates dataset (local blank templates) ─────────────────
if TEMPLATE_DIR.exists():
    exts = ['*.jpg','*.jpeg','*.png','*.webp']
    tfiles = []
    for ext in exts:
        tfiles.extend(glob.glob(str(TEMPLATE_DIR/'**'/ext), recursive=True))
    for fp in tfiles:
        name = Path(fp).stem.replace('_',' ').replace('-',' ').title()
        MEME_TEMPLATES.append({'name': name, 'path': fp, 'url': None, 'source': 'kaggle-templates'})
    print(f'📦 kaggle-templates: {len(tfiles)} images loaded')

# ── Source 2: memotion-dataset-7k (memes with text labels) ───────────────────
if MEMOTION_DIR.exists():
    csvs = list(MEMOTION_DIR.glob('**/*.csv'))
    if csvs:
        df = pd.read_csv(csvs[0])
        print(f'📦 memotion: {len(df)} rows | cols: {list(df.columns)}')
        img_col   = next((c for c in df.columns if 'image' in c.lower() or 'file' in c.lower()), None)
        label_col = next((c for c in df.columns if 'text' in c.lower() or 'caption' in c.lower()), None)
        if img_col:
            n = 0
            for _, row in df.iterrows():
                for subdir in ['images','7k dataset','data','']:
                    p = MEMOTION_DIR / subdir / str(row[img_col]) if subdir else MEMOTION_DIR / str(row[img_col])
                    if p.exists():
                        lbl = str(row[label_col])[:60] if label_col else p.stem
                        MEME_TEMPLATES.append({'name': lbl, 'path': str(p), 'url': None, 'source': 'memotion'})
                        n += 1
                        break
            print(f'   Added {n} memotion images')

# ── Source 3: reddit-memes-dataset ───────────────────────────────────────────
if REDDIT_DIR.exists():
    rcsvs = list(REDDIT_DIR.glob('**/*.csv'))
    if rcsvs:
        rd = pd.read_csv(rcsvs[0])
        print(f'📦 reddit-memes: {len(rd)} rows | cols: {list(rd.columns)}')
        title_col = next((c for c in rd.columns if 'title' in c.lower() or 'name' in c.lower()), None)
        url_col   = next((c for c in rd.columns if 'url' in c.lower()), None)
        if title_col and url_col:
            for _, row in rd.head(200).iterrows():
                MEME_TEMPLATES.append({'name': str(row[title_col])[:60], 'path': None,
                                       'url': str(row[url_col]), 'source': 'reddit'})
            print(f'   Added {min(200, len(rd))} reddit meme URLs')

# ── Source 4: imgflip API fallback (always works) ─────────────────────────────
try:
    r = requests.get('https://api.imgflip.com/get_memes', timeout=10)
    for m in r.json()['data']['memes']:
        MEME_TEMPLATES.append({'name': m['name'], 'path': None, 'url': m['url'], 'source': 'imgflip'})
    print(f'🌐 imgflip API: {len(r.json()["data"]["memes"])} templates added')
except Exception as e:
    print(f'⚠️ imgflip API failed: {e}')

print(f'\n✅ Total templates: {len(MEME_TEMPLATES)}')

# Save URL-based templates for HuggingFace deployment
hf_templates = [t for t in MEME_TEMPLATES if t['url']]
with open(EXPORT_DIR / 'meme_templates.json', 'w') as f:
    json.dump(hf_templates, f, indent=2)
print(f'💾 Saved {len(hf_templates)} URL-templates → {EXPORT_DIR}/meme_templates.json')

In [ ]:
template_names = [t['name'] for t in MEME_TEMPLATES]
vectorizer    = TfidfVectorizer(ngram_range=(1,2), min_df=1)
tfidf_matrix  = vectorizer.fit_transform(template_names)


def find_best_template(context: str, top_k: int = 5) -> list:
    """Return top-k (template_dict, score) tuples for a text context."""
    qvec   = vectorizer.transform([context])
    scores = cosine_similarity(qvec, tfidf_matrix).flatten()
    idxs   = scores.argsort()[::-1][:top_k * 4]
    results = [(MEME_TEMPLATES[i], float(scores[i])) for i in idxs]

    # Prefer local Kaggle files first, then URL-based
    local = [(t,s) for t,s in results if t['path'] and Path(t['path']).exists()]
    urls  = [(t,s) for t,s in results if t['url']]
    combined = (local + urls)[:top_k]

    if not combined:
        # Full random fallback
        pop = [t for t in MEME_TEMPLATES if t['url']]
        combined = [(pop[i], 0.0) for i in np.random.choice(len(pop), min(top_k,len(pop)), replace=False)]
    return combined


def load_template_image(template: dict) -> Image.Image:
    """Load a meme template from local path or URL."""
    if template.get('path') and Path(template['path']).exists():
        return Image.open(template['path']).convert('RGB')
    elif template.get('url'):
        resp = requests.get(template['url'], timeout=15)
        return Image.open(BytesIO(resp.content)).convert('RGB')
    raise ValueError(f'No valid path or URL for: {template["name"]}')


# Test
matches = find_best_template('Monday mornings and coffee')
print('Test matches:')
for t, s in matches:
    print(f'  [{s:.3f}] "{t["name"]}"  ({t["source"]})')

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Device: {DEVICE}  dtype: {DTYPE}')

#
# BLIP-2 options by GPU VRAM:
#   Salesforce/blip2-opt-2.7b       ->  ~6 GB   <-- default
#   Salesforce/blip2-flan-t5-xl     -> ~12 GB   (better reasoning)
#   Salesforce/blip2-flan-t5-xxl    -> ~24 GB   (needs A100)
#
BLIP2_ID        = 'Salesforce/blip2-opt-2.7b'
BLIP2_SAVE_PATH = EXPORT_DIR / 'blip2'

print(f'\n⏳ Loading {BLIP2_ID}...')
blip2_processor = Blip2Processor.from_pretrained(BLIP2_ID)
blip2_model     = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_ID, torch_dtype=DTYPE, device_map='auto'
)
blip2_model.eval()
print(f'✅ BLIP-2 loaded')

print(f'\n💾 Saving BLIP-2 → {BLIP2_SAVE_PATH}  ...')
blip2_processor.save_pretrained(str(BLIP2_SAVE_PATH))
blip2_model.save_pretrained(str(BLIP2_SAVE_PATH))
sz = sum(f.stat().st_size for f in BLIP2_SAVE_PATH.rglob('*') if f.is_file())
print(f'✅ BLIP-2 saved  ({sz/1e9:.2f} GB)')